In [1]:
!pip install -q kaggle

In [3]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_bc4bab73784a391f22d4b2a64f123939"

In [4]:
!kaggle datasets list -s food-101

ref                                                             title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------  ------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
nehaprabhavalkar/indian-food-101                                Indian Food 101                                          6951  2020-09-30 06:23:43.063000          40406        627                1  
kmader/food41                                                   Food Images (Food-101)                             5693808106  2018-05-15 07:40:37.080000          63076        766           0.8125  
dansbecker/food-101                                             Food 101                                          10067059132  2018-01-03 20:44:28.610000          57130        314           0.6875  
kuant

In [5]:
!kaggle datasets files -d msarmi9/food101tiny

Next Page Token = CfDJ8LXGgraiXMNClPzcySXY5U5WywEMhZ7w9D0DMK8QQyeSzlI5WaQaOFvQ73fffpOsUxbwrfQVXOVzhVORby5mayngrWy7opAslxObtvFy1CunsXSnt2M1s497I53J8EUhbogodQSvgh_pwqXs2Hrf9Nv-OdZsk7CAZnuewOsiWePHrd7IUg
name                                             size  creationDate                
----------------------------------------------  -----  --------------------------  
data/food-101-tiny/train/apple_pie/1005649.jpg  42196  2020-07-21 02:33:04.163000  
data/food-101-tiny/train/apple_pie/1014775.jpg  48051  2020-07-21 02:33:04.046000  
data/food-101-tiny/train/apple_pie/1026328.jpg  59055  2020-07-21 02:33:04.104000  
data/food-101-tiny/train/apple_pie/1028787.jpg  52592  2020-07-21 02:33:04.097000  
data/food-101-tiny/train/apple_pie/1043283.jpg  45313  2020-07-21 02:33:04.110000  
data/food-101-tiny/train/apple_pie/1050519.jpg  65700  2020-07-21 02:33:03.955000  
data/food-101-tiny/train/apple_pie/1057749.jpg  41541  2020-07-21 02:33:04.081000  
data/food-101-tiny/train/apple_pie/1057810.

In [6]:
!kaggle datasets download -d msarmi9/food101tiny

Dataset URL: https://www.kaggle.com/datasets/msarmi9/food101tiny
License(s): copyright-authors
100% 96.1M/96.1M [00:00<00:00, 129MB/s]



In [7]:
import zipfile
import os

with zipfile.ZipFile("food101tiny.zip", "r") as zip_ref:
    zip_ref.extractall("/content/food101")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [8]:
import os

print(os.listdir("/content/food101"))

['data']


In [9]:
import os

dataset_path = "/content/food101/data/food-101-tiny"

print("Folders:")
print(os.listdir(dataset_path))

train_path = os.path.join(dataset_path, "train")

classes = os.listdir(train_path)

print("\nNumber of classes:", len(classes))
print("First 10 classes:", classes[:10])

Folders:
['valid', 'train']

Number of classes: 10
First 10 classes: ['ice_cream', 'tiramisu', 'edamame', 'ramen', 'sushi', 'falafel', 'french_toast', 'apple_pie', 'cannoli', 'bibimbap']


In [10]:
import os

dataset_path = "/content/food101/data/food-101-tiny"

train_path = os.path.join(dataset_path, "train")
valid_path = os.path.join(dataset_path, "valid")

print("TRAIN IMAGES")
for class_name in sorted(os.listdir(train_path)):
    class_path = os.path.join(train_path, class_name)
    count = len(os.listdir(class_path))
    print(f"{class_name}: {count}")

print("\nVALIDATION IMAGES")
for class_name in sorted(os.listdir(valid_path)):
    class_path = os.path.join(valid_path, class_name)
    count = len(os.listdir(class_path))
    print(f"{class_name}: {count}")

TRAIN IMAGES
apple_pie: 150
bibimbap: 150
cannoli: 150
edamame: 150
falafel: 150
french_toast: 150
ice_cream: 150
ramen: 150
sushi: 150
tiramisu: 150

VALIDATION IMAGES
apple_pie: 50
bibimbap: 50
cannoli: 50
edamame: 50
falafel: 50
french_toast: 50
ice_cream: 50
ramen: 50
sushi: 50
tiramisu: 50


In [11]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

dataset_path = "/content/food101/data/food-101-tiny"

train_path = dataset_path + "/train"
valid_path = dataset_path + "/valid"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

valid_datagen = ImageDataGenerator(
    rescale=1./255
)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

valid_data = valid_datagen.flow_from_directory(
    valid_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

print("Classes:", train_data.class_indices)

Found 1500 images belonging to 10 classes.
Found 500 images belonging to 10 classes.
Classes: {'apple_pie': 0, 'bibimbap': 1, 'cannoli': 2, 'edamame': 3, 'falafel': 4, 'french_toast': 5, 'ice_cream': 6, 'ramen': 7, 'sushi': 8, 'tiramisu': 9}


In [12]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

# Load MobileNetV2 without the final classification layer
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the pretrained layers
base_model.trainable = False

# Add our own classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(10, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

# Compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        "best_food_model.keras",
        monitor="val_accuracy",
        save_best_only=True
    )
]

history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.4180 - loss: 1.6583 - val_accuracy: 0.7100 - val_loss: 0.9741
Epoch 2/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 123s 3s/step - accuracy: 0.6393 - loss: 1.0682 - val_accuracy: 0.7600 - val_loss: 0.7782
Epoch 3/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - accuracy: 0.6920 - loss: 0.8971 - val_accuracy: 0.7560 - val_loss: 0.7017
Epoch 4/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - accuracy: 0.7633 - loss: 0.7287 - val_accuracy: 0.7700 - val_loss: 0.6571
Epoch 5/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 114s 2s/step - accuracy: 0.7747 - loss: 0.6739 - val_accuracy: 0.7920 - val_loss: 0.6108
Epoch 6/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 104s 2s/step - accuracy: 0.7907 - loss: 0.6036 - val_accuracy: 0.7720 - val_loss: 0.6444
Epoch 7/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - accuracy: 0.7980 - loss: 0.5979 - val_accuracy: 0.7920 - val_loss: 0.6220
Epoch 8/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 104s 2s/step - accuracy: 0.8107 - loss: 0.5466 - val_accuracy: 0.8060 - v

In [14]:
loss, accuracy = model.evaluate(valid_data)

print("Validation Loss:", round(loss, 4))
print("Validation Accuracy:", round(accuracy * 100, 2), "%")

16/16 ━━━━━━━━━━━━━━━━━━━━ 33s 2s/step - accuracy: 0.8060 - loss: 0.6373
Validation Loss: 0.6373
Validation Accuracy: 80.6 %


In [15]:
model.save("/content/best_food_model.keras")

print("Model saved successfully!")

Model saved successfully!


In [18]:
from tensorflow.keras.preprocessing import image
import numpy as np

def predict_food(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0

    predictions = model.predict(img_array, verbose=0)

    predicted_index = np.argmax(predictions[0])
    predicted_class = list(train_data.class_indices.keys())[predicted_index]
    confidence = predictions[0][predicted_index] * 100

    print("Predicted Food:", predicted_class)
    print("Confidence:", round(confidence, 2), "%")

In [20]:
from google.colab import files

uploaded = files.upload()

image_path = list(uploaded.keys())[0]

print("Image uploaded:", image_path)

Saving ice cream food task.webp to ice cream food task.webp
Image uploaded: ice cream food task.webp


In [21]:
predict_food(image_path)

Predicted Food: ice_cream
Confidence: 72.05 %


In [22]:
calorie_data = {
    "apple_pie": 300,
    "bibimbap": 500,
    "cannoli": 200,
    "edamame": 120,
    "falafel": 330,
    "french_toast": 180,
    "ice_cream": 200,
    "ramen": 500,
    "sushi": 300,
    "tiramisu": 450
}

predicted_food = "ice_cream"

print("Food:", predicted_food)
print("Estimated Calories:", calorie_data[predicted_food], "kcal")

Food: ice_cream
Estimated Calories: 200 kcal


In [23]:
from google.colab import files
from tensorflow.keras.preprocessing import image
import numpy as np

# Calorie information (approximate kcal per typical serving)
calorie_data = {
    "apple_pie": 300,
    "bibimbap": 500,
    "cannoli": 200,
    "edamame": 120,
    "falafel": 330,
    "french_toast": 180,
    "ice_cream": 200,
    "ramen": 500,
    "sushi": 300,
    "tiramisu": 450
}

# Upload food image
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# Load and preprocess image
img = image.load_img(image_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict food
predictions = model.predict(img_array, verbose=0)

predicted_index = np.argmax(predictions[0])
class_names = list(train_data.class_indices.keys())

predicted_food = class_names[predicted_index]
confidence = predictions[0][predicted_index] * 100

# Get calorie estimate
estimated_calories = calorie_data[predicted_food]

# Display result
print("=" * 40)
print("       FOOD RECOGNITION RESULT")
print("=" * 40)
print("Food:", predicted_food.replace("_", " ").title())
print("Confidence:", round(confidence, 2), "%")
print("Estimated Calories:", estimated_calories, "kcal")
print("(Approximate per typical serving)")
print("=" * 40)

Saving ice cream food task.webp to ice cream food task (1).webp
       FOOD RECOGNITION RESULT
Food: Ice Cream
Confidence: 72.05 %
Estimated Calories: 200 kcal
(Approximate per typical serving)
